# 01. Linear Regression - California Housing Prices

## What is this notebook about?

In this notebook we predict the **median value of houses** in California districts using **Linear Regression**.

Linear Regression is the simplest possible ML algorithm. It draws the best straight line (or hyperplane in multiple dimensions) through your data and uses that line to predict a number.

## What you will learn

1. How to **load** a real dataset
2. How to **explore** it with summary stats and plots
3. How to **train** a simple linear regression on one feature
4. How to **evaluate** a regression model with MAE, RMSE, and R²
5. How to extend to **multiple features** and **interpret the coefficients**

## Dataset

We use the **California Housing** dataset built into scikit-learn (no download needed).
- 20,640 rows (each row = one census district)
- 8 features (income, age of houses, location, etc.)
- Target: `MedHouseVal` (median house value, in $100,000s)

Let's get started!

In [ ]:
# If you're running locally, uncomment and run this once.
# In Google Colab, all of these are pre-installed - you can skip this cell.
# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
# Standard imports used in every ML notebook
import numpy as np                 # numerical arrays
import pandas as pd                # tabular data (DataFrames)
import matplotlib.pyplot as plt    # plotting
import seaborn as sns              # prettier statistical plots

# Set up plot style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Set a random seed so results are reproducible
np.random.seed(42)

## Step 1. Load the data

The dataset is included in scikit-learn. We just need to call one function. Setting `as_frame=True` returns it as a pandas DataFrame, which is easier to explore.

In [ ]:
from sklearn.datasets import fetch_california_housing

# Load the dataset (returns features X and target y as a single DataFrame)
data = fetch_california_housing(as_frame=True)
df = data.frame                # the full DataFrame with both X and y

print(f"Dataset shape: {df.shape}")  # (rows, columns)
df.head()                            # show the first 5 rows

**What just happened?**
We loaded a DataFrame `df` with 20,640 rows and 9 columns. The first 8 columns are features. The last column `MedHouseVal` is the target we want to predict.

Each row is one census district in California.

In [ ]:
# .info() shows the data type and missing-value count of each column
print(df.info())

# .describe() gives summary statistics (mean, std, min, max, quartiles)
df.describe()

## Step 2. Explore the data

Before modelling, **always look at your data**. We will:
1. Plot the distribution of the target.
2. Check correlations between features.
3. Plot the strongest predictor against the target.

In [ ]:
# Distribution of the target value (price)
plt.figure(figsize=(10, 5))
sns.histplot(df["MedHouseVal"], bins=50, kde=True, color="gray")
plt.title("Distribution of Median House Value")
plt.xlabel("Median House Value ($100,000s)")
plt.show()

**Notice:** the distribution is right-skewed and there is a spike at 5.0 - that's the maximum value (anything above $500k was capped). This is a real-world data quirk worth being aware of.

In [ ]:
# Heatmap of how each pair of features correlates
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

**What to look for:** the bottom row (correlation with `MedHouseVal`) tells you which features are best for prediction. `MedInc` (median income) jumps out at +0.69 - far stronger than anything else.

In [ ]:
# Scatter plot of the strongest predictor against the target
# Use a sample to avoid plotting 20,000 dots
plt.figure(figsize=(10, 6))
sns.scatterplot(x="MedInc", y="MedHouseVal", data=df.sample(2000), alpha=0.4, color="steelblue")
plt.title("Median Income vs Median House Value")
plt.show()

You can see a clear upward trend: higher income areas tend to have more expensive houses. A straight line through this scatter would be a reasonable predictor.

## Step 3. Simple Linear Regression (one feature)

Let's fit a straight line using **only `MedInc`** as the predictor.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# X is the feature(s), y is the target
X = df[["MedInc"]]               # double brackets keep it as a DataFrame
y = df["MedHouseVal"]

# Split into 80% training data, 20% test data
# random_state=42 makes the split reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create the model and fit it on the training data
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions on the test data
y_pred = model.predict(X_test)

# Print the learned line: y = slope * x + intercept
print(f"Slope (coefficient):  {model.coef_[0]:.4f}")
print(f"Intercept:            {model.intercept_:.4f}")

# Evaluate the predictions on the test set
print(f"\nMAE:   {mean_absolute_error(y_test, y_pred):.4f}    (avg error in $100k)")
print(f"RMSE:  {mean_squared_error(y_test, y_pred, squared=False):.4f}    (penalizes big errors more)")
print(f"R^2:   {r2_score(y_test, y_pred):.4f}    (1.0 = perfect, 0 = no better than mean)")

**Reading the results:**
- **Slope ≈ 0.42** means: each $10k increase in median income corresponds to a $42k increase in predicted house value.
- **R² ≈ 0.47** means our one-variable model captures about 47% of the variance. Not bad, but we can do better with more features.

In [ ]:
# Plot: actual values (dots) vs the fitted line (red)
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_test["MedInc"], y=y_test, alpha=0.3, color="steelblue", label="Actual")
sns.lineplot(x=X_test["MedInc"], y=y_pred, color="red", label="Predicted (line)")
plt.title("Linear Regression Fit: Income vs House Value")
plt.xlabel("Median Income")
plt.ylabel("Median House Value")
plt.legend()
plt.show()

## Step 4. Multiple Linear Regression (all features)

Now let's use **all 8 features** instead of just one. The math is the same: we still find the best linear combination, but now in 8-dimensional space.

In [ ]:
# Use all features except the target
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

# Re-split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a new linear regression model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Evaluate
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE: {mean_squared_error(y_test, y_pred, squared=False):.4f}")
print(f"R^2:  {r2_score(y_test, y_pred):.4f}")

**R² should jump from ~0.47 to ~0.60.** Adding more features helped a lot.

Now let's see which features mattered most by inspecting the learned coefficients.

In [ ]:
# Coefficients tell us how each feature contributes to the prediction
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_,
}).sort_values("Coefficient", key=abs, ascending=False)  # sort by absolute size

print(coef_df)

**Interpretation:**
- A **positive** coefficient means more of that feature → higher predicted price.
- A **negative** coefficient means more of that feature → lower predicted price.
- The **bigger the absolute value**, the more influence the feature has (assuming features are on similar scales).

In [ ]:
# Predicted vs Actual plot - the closer to the diagonal line, the better
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.3, color="steelblue")
plt.plot([0, 5], [0, 5], color="red", linestyle="--", label="Perfect prediction")
plt.xlabel("Actual House Value")
plt.ylabel("Predicted House Value")
plt.title("Predicted vs Actual House Values")
plt.legend()
plt.show()

## Step 5. Exercises - try these on your own!

1. **Add polynomial features:**
   ```python
   from sklearn.preprocessing import PolynomialFeatures
   poly = PolynomialFeatures(degree=2)
   X_poly = poly.fit_transform(X_train)
   ```
   Does R² improve?

2. **Drop the most correlated features** (`AveRooms` and `AveBedrms` are highly correlated). Does it hurt much?

3. **Plot residuals**: `residuals = y_test - y_pred`. Are they centered around 0? Any patterns? Patterns suggest a non-linear effect you missed.

4. **Compute Adjusted R² manually**:
   `Adj R² = 1 - (1 - R²) * (n - 1) / (n - p - 1)` where n = sample size, p = number of features.

5. **Try a Kaggle dataset** (e.g. House Prices: Advanced Regression Techniques). The workflow is the same.

When you are done, head to **02 - Logistic Regression**!